# Semana 01 — Lab 02: `sklearn` Pipeline + validación cruzada + PR-AUC (AP)

## Contexto
En el Lab NumPy construimos el workflow “sin magia” para entender:
- split train/val/test
- métricas (matriz de confusión, accuracy, precisión, recall)
- curva PR y AP
- fuga por preprocesamiento

Ahora damos el salto a la **práctica estándar** con `scikit-learn`, cuidando que el pipeline sea **reproducible** y **sin fuga**.

## Objetivo
Implementar un flujo de entrenamiento/evaluación con:
- `Pipeline(StandardScaler, LogisticRegression)`
- validación cruzada (K-fold)
- métricas: `accuracy` y `average_precision` (AP / PR-AUC)
- auditoría final en test con curva PR

## Regla práctica
- **Decisiones** (modelo, hiperparámetros, umbral) con validación/CV.
- **Test** se usa solo para auditoría final (una vez).


## Pseudocódigo (workflow correcto)

```text
Entrada:
  D = {(x_i, y_i)}           # datos
  métrica ∈ {accuracy, AP}   # según el problema
  Grid(λ)                    # hiperparámetros (p.ej. C o λ)
  K                          # folds

Paso 0: separar test (auditoría final)
  (D_trainval, D_test) = split(D)

Paso 1: definir pipeline (evita fuga)
  pipe = Pipeline([
    StandardScaler(),
    LogisticRegression(λ)
  ])

Paso 2: selección con validación cruzada (NO tocar test)
  para cada λ en Grid(λ):
      scores_λ = CV_score(pipe(λ), D_trainval, scoring=métrica, K)
  λ* = argmax_λ mean(scores_λ)

Paso 3: entrenar final en trainval
  pipe(λ*).fit(D_trainval.X, D_trainval.y)

Paso 4: auditar en test (una sola vez)
  scores = pipe(λ*).predict_proba(D_test.X)   # o decision_function
  reportar:
      accuracy(D_test.y, scores, umbral t)
      precision_recall_curve(D_test.y, scores)
      AP(D_test.y, scores)

Salida:
  métrica_CV(λ*), métricas_test, curva PR, AP
```

> Nota: si ajustas \(\lambda\) o el umbral \(t\) mirando test, introduces sesgo por selección.


In [ ]:
# 0) Setup

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, average_precision_score, accuracy_score

rng = np.random.default_rng(0)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})


## 1) Datos y split (test como auditoría)

Usaremos un dataset binario clásico (cáncer de mama). Primero separamos un **test** fijo para auditoría final.

- `trainval`: se usa para CV y selección de hiperparámetros
- `test`: se usa una sola vez al final


In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target  # 0/1

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print("shapes:", X_trainval.shape, X_test.shape)
print("prevalencia y=1 (trainval/test):", y_trainval.mean().round(3), y_test.mean().round(3))


## 2) Pipeline (evitar fuga)

**Por qué pipeline:** `StandardScaler.fit` debe aprenderse **solo** con datos de entrenamiento.

- Sin pipeline, es fácil cometer fuga al estandarizar con todo el dataset.
- Con pipeline + CV, cada fold ajusta el escalador **dentro del fold**.

Tarea:
- construye el pipeline `StandardScaler → LogisticRegression`.


In [ ]:
# TODO (alumno): puedes cambiar hiperparámetros de LogisticRegression aquí
# Nota: usamos solver estable y max_iter alto para evitar warning de convergencia

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, solver="liblinear"))
])

pipe


## 3) Validación cruzada: estimar desempeño (accuracy vs AP)

Haremos K-fold estratificado para obtener una estimación más estable.

Tareas:
- calcula `cross_val_score(..., scoring="accuracy")`
- calcula `cross_val_score(..., scoring="average_precision")`  (AP / PR-AUC)

Pregunta: ¿por qué AP suele ser más informativa cuando hay desbalance?


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scores_acc = cross_val_score(pipe, X_trainval, y_trainval, cv=cv, scoring="accuracy")
scores_ap = cross_val_score(pipe, X_trainval, y_trainval, cv=cv, scoring="average_precision")

print("CV accuracy: mean ± std =", scores_acc.mean().round(3), "±", scores_acc.std().round(3))
print("CV AP:       mean ± std =", scores_ap.mean().round(3), "±", scores_ap.std().round(3))


## 4) (Opcional) Selección de hiperparámetro con CV

Vamos a barrer valores de `C` (inverso de regularización) y elegir el que maximiza la métrica promedio.

> Regla: este tipo de decisiones se hacen con CV en `trainval`, no con `test`.


In [ ]:
Cs = np.logspace(-3, 2, 8)
mean_ap = []

for C in Cs:
    pipe_C = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=5000, solver="liblinear"))
    ])
    s = cross_val_score(pipe_C, X_trainval, y_trainval, cv=cv, scoring="average_precision")
    mean_ap.append(s.mean())

mean_ap = np.array(mean_ap)
C_star = float(Cs[int(np.argmax(mean_ap))])
print("C grid:", Cs)
print("mean AP:", np.round(mean_ap, 3))
print("C* (max mean AP):", C_star)


## 5) Entrenar en trainval y auditar en test

Entrenamos el pipeline final con \(C^\*\) (si hiciste el paso opcional) y evaluamos en test:

- **accuracy** (a un umbral implícito 0.5)
- **curva PR** (variando umbral)
- **AP** (precisión promedio)

> Nota: la curva PR se construye con scores (probabilidades) y al variar el umbral.


In [ ]:
pipe_final = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=C_star, max_iter=5000, solver="liblinear"))
])

pipe_final.fit(X_trainval, y_trainval)

# Scores (probabilidad de clase positiva)
scores = pipe_final.predict_proba(X_test)[:, 1]

# Accuracy a umbral 0.5
pred_05 = (scores >= 0.5).astype(int)
acc_test = accuracy_score(y_test, pred_05)

# PR curve + AP
prec, rec, thr = precision_recall_curve(y_test, scores)
ap_test = average_precision_score(y_test, scores)

print("TEST accuracy (t=0.5):", round(acc_test, 3))
print("TEST AP:", round(ap_test, 3))

plt.figure()
plt.plot(rec, prec, linewidth=2)
plt.axhline(y_test.mean(), linestyle="--", color="black", linewidth=1.5,
            label=f"línea base ≈ prevalencia π={y_test.mean():.2f}")
plt.xlabel("recall")
plt.ylabel("precisión")
plt.title(f"Curva Precisión–Recall (AP = {ap_test:.3f})")
plt.legend()
plt.tight_layout()
plt.show()


## 6) Entregable (celda final)

Incluye al final:
- `mean±std` de CV para **accuracy** y **AP**
- tu elección de \(C^\*\) (si hiciste el barrido)
- curva PR en test + valor de **AP**
- 4–6 líneas:
  - ¿por qué un pipeline evita fuga?
  - ¿por qué CV ayuda (varianza del split)?
  - ¿cuándo preferirías AP sobre accuracy?
